## 1. Prepare all Nigeria H3 Cells  at Resolution 8

### 00. Step Up

In [ ]:
%load_ext autoreload
%autoreload 2 

import sys
from pathlib import Path 
import logging 
from datetime import datetime
import pickle
import h3
from codebase.utils.utils import setup_logging 
import sys 
from config.settings import STORAGE_CONFIG, ADMIN_DATA_SOURCES, INPUT_BASE_DATA_SOURCES, RAW_DATA_DIR,PROCESSED_DATA_DIR, EXPORTS_DIR, OUTPUT_DIR
import pandas as pd
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 


logger = setup_logging(log_dir='log-main-sp-clustering-and-routing', 
                       projname='log-main-spcr')

### 01. Load and Preprocess Boundary Data

In [ ]:
%load_ext autoreload
%autoreload 2 


import sys
from pathlib import Path
project_root = Path('').parent
sys.path.append(str(project_root))


from pathlib import Path 
import logging 
from datetime import datetime
import pickle
from src.h3_spatial_system.data.downloader import DataDownloader, validate_and_summarize_data
from src.h3_spatial_system.h3_system.generator import H3AddressGenerator
from src.h3_spatial_system.storage.duckdb_storage import DuckDBStorage
from config.settings import * 


validate_downloaded_data with the function validate_and_summarize_data save the standardized geojson file to disk

In [ ]:
# Step 1: Download administrative boundary data
'''
logger.info("📥 Step 1: Downloading administrative boundary data...")
downloader = DataDownloader()
downloaded_files = downloader.download_admin_boundaries()
if downloaded_files:
    validation_results, summaries = validate_and_summarize_data(downloaded_files)
    
if not downloaded_files:
    logger.error("❌ Failed to download administrative boundary data")
    # return 1

logger.info(f"✅ Downloaded {len(downloaded_files)} boundary files")

'''

# load geojson files
dict_path_geojson ={
    'states': RAW_DATA_DIR / 'grid3-nga-operational-state-boundaries_standardized.geojson',
    'lgas': RAW_DATA_DIR / 'grid3-nga-operational-lga-boundaries_standardized.geojson',
    'wards': RAW_DATA_DIR / 'grid3-nga-operational-wards-v1-0_standardized.geojson'
}

logger.info("🚀 Starting H3-based address system generation for Nigeria")

# Step 2: Initialize H3 address generator
logger.info("🔧 Step 2: Initializing H3 address generator...")
generator = H3AddressGenerator(resolution=H3_RESOLUTION)

In [ ]:
# Step 3: Load administrative boundaries
logger.info("🗺️ Step 3: Loading administrative boundaries...")
states_path = str(dict_path_geojson['states'])
lgas_path = str(dict_path_geojson['lgas'])
wards_path = str(dict_path_geojson['wards'])

generator.load_admin_boundaries(states_path, lgas_path, wards_path)
logger.info("✅ Administrative boundaries loaded")

In [ ]:
# ADMIN_DATA_SOURCES
str({key:value['standardize_file_path'] for key, value in ADMIN_DATA_SOURCES.items()}['states'])

### 02. Generate H3 Cells

In [ ]:
# Step 4: Generate H3 cells
logger.info("🔷 Step 4: Generating H3 cells...")
h3_cells = generator.generate_h3_cells() 
with open(PROCESSED_DATA_DIR'h3_cells_res8.pickle', 'wb') as f:
    pickle.dump(h3_cells, f)

### 03. Add Meta data to H3 Cells

In [ ]:
%load_ext autoreload
%autoreload 2 


import sys
from pathlib import Path
import logging 

import h3
import pickle
from src.h3_spatial_system.h3_system.utils import h3_to_objects_parallel_safe, h3_to_objects_parallel_generator, H3SQLiteManager

In [ ]:
# Load Data
with open('./data/processed/h3_cells_res8.pickle', 'rb') as f:
    h3_cells = pickle.load(f)
logger.info(f"✅ Generated {len(h3_cells):,} H3 cells")

In [ ]:
# import time 
# start_time = time.time()   
# h3_cells_processed =  h3_to_objects_parallel_safe(h3_cells[0:1000]) 
# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 

In [ ]:
# # For very large datasets, use batched processing:
# h3_data = {}
# for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
#     h3_data[h3_index] = result ## 9m 17s

# with open('./data/processed/h3_cells_processed_dict.pickle', 'wb') as f:
#     pickle.dump(h3_data, f)



# METHOD 2
# Stream directly to JSONL - most efficient
# Process and save your H3 data
# import time 
# start_time = time.time()   
# with H3SQLiteManager("./data/processed/h3_data.db") as db:
#     for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
#         db.add_result(h3_index, result)

# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 
# print("Done! Database saved with compression.")

In [ ]:
from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_table_and_h3_cells_data    
from config.settings import STORAGE_CONFIG

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']


# METHOD 3
# Stream directly to DUCKDB - most efficient
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH, batch_size=100000) as db:
    count = 0
    for h3_index, result in h3_to_objects_parallel_generator(h3_cells):
        db.add_result(h3_index, result)
        count += 1
        
        # Check database periodically to verify commits
        if count % 100000 == 0:
            stats = db.get_stats()
            print(f"Processed: {count:,}, In DB: {stats['total_records']:,}")

# # After the context manager exits, check final state
# verify_db_table_and_h3_cells_data(db_path)

In [ ]:
# Check if your current database file exists and has data
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_table_and_h3_cells_data
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 
verify_db_table_and_h3_cells_data(H3_DUCKDB_PATH)

In [ ]:
# print(len(h3_data))
# print(len(h3_data.keys()))

### 04. Generate Address and Address ID for the Cells & ADD TO DB

In [ ]:
%load_ext autoreload
%autoreload 2 

from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
from src.h3_spatial_system.h3_system.save_h3_data_utils import verify_db_table_and_h3_cells_data   
from config.settings import STORAGE_CONFIG 

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

In [ ]:
# # Step 5: Generate addresses - to-do optu
# start_time = time.time() 
# logger.info("🏠 Step 5: Generating addresses...") 
# addresses = generator.generate_addresses(h3_cells) 
# logger.info(f"✅ Generated {len(addresses)} address records") 
# parallel_time = time.time() - start_time 
# print(f"Total time taken {len(h3_cells):,} cells took: {parallel_time:.2f} seconds") 


# To be Implemented
# This can handle both new and existing H3 indices
# with FastH3DuckDBManager(resolution=8, db_path=db_path) as db:
    # address_data = generator.generate_addresses(h3_cells) 
    # db.upsert_address_data(address_data)

In [ ]:
# Verifing DB
verify_db_table_and_h3_cells_data(H3_DUCKDB_PATH)

In [ ]:
# Added New addrss columns || Alternatively I can use raw sql
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db:
    db._add_address_columns()

In [ ]:
import duckdb 
conn = duckdb.connect(H3_DUCKDB_PATH)

In [ ]:
conn.execute("SELECT count(distinct h3_index) FROM h3_cells where state_code is NULL").fetchdf()  #803,827

In [ ]:
import pandas as pd
PATH_H3_ADDRESS_DF = EXPORTS_DIR/ 'df_filterd_h3_res8_nigeria_addresses.parquet' 

df_address_filtered = duckdb.sql(f"SELECT * EXCLUDE __index_level_0__ FROM '{PATH_H3_ADDRESS_DF}'").fetchdf() 
df_address_filtered.rename(columns={'h3_id':'h3_index'}, inplace=True)
df_address_filtered['resolution'] = 8 
df_address_filtered.head(1)
print(f'Total Records: {len(df_address_filtered):,}')

In [ ]:
# Set your chunk size
batch_size = 100000
total_processed = 0

# Loop through the DataFrame in chunks
for i in range(0, len(df_address_filtered), batch_size):
    batch_data = df_address_filtered.iloc[i:i + batch_size]
    # print(f"Chunk {i // batch_size + 1}:\n", batch_data.head(), "\n")

    try: 
        conn.register('address_upsert_df', batch_data) 
        # Use INSERT OR REPLACE for UPSERT
        conn.execute("""
            INSERT OR REPLACE INTO h3_cells (
                h3_index, resolution, 
                h3_derived_id, grid_position_id, primary_address_id,
                country_code, country_name, state_code, state_name,
                lga_code, lga_name, ward_code, ward_name,
                confidence_level, coverage_percentage, area_km2
            )
            SELECT 
                h3_index, resolution,
                h3_derived_id, grid_position_id, primary_address_id,
                country_code, country_name, state_code, state_name,
                lga_code, lga_name, ward_code, ward_name,
                confidence_level, coverage_percentage, area_km2
            FROM address_upsert_df
        """)
        
        conn.unregister('address_upsert_df')
        total_processed += len(batch_data)
        
        print(f"✅ Upserted batch {i//batch_size + 1}: {len(batch_data):,} records (total: {total_processed:,})")
        
    except Exception as e:
        print(f"❌ Error upserting : {e}")
        # print(f"❌ Error upserting batch {i//batch_size + 1}: {e}")
        # continue
    

# print(f"🎉 Address data upsert completed: {total_processed:,} records processed")

In [ ]:
conn.close()

### 05. Evaluation with Plotting

In [ ]:
# pip install shiny shinywidgets hvplot geoviews geopandas hvplot holoviews bokeh shapely

In [ ]:
%load_ext autoreload
%autoreload 2 


import duckdb 
import geopandas as gpd
import pandas as pd
from pathlib import Path
from config.settings import STORAGE_CONFIG 
from src.h3_system.plot_utils import plot_h3_from_db #, plot_h3_from_db_fast

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
test_map_path = Path('./output/map/test/')

In [ ]:
conn = duckdb.connect(H3_DUCKDB_PATH)
# Schema
# conn.execute('DESCRIBE h3_cells;').fetchdf()[['column_name', 'column_type', 'null', 'key']]

In [ ]:
df_sample_h3_with_metadata = gpd.GeoDataFrame(conn.execute('SELECT * FROM h3_cells WHERE confidence_level IS NOT NULL LIMIT 100').fetch_df())
# print(df_samole_h3_with_metadata.columns)
df_sample_h3_with_metadata.sample(2)

# ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 'polygon_wkt',
# 'boundary_json', 'latlng_json', 'polygon_area', 'num_vertices', 'error',
# 'created_at', 'h3_derived_id', 'grid_position_id', 'primary_address_id',
# 'country_code', 'country_name', 'state_code', 'state_name', 'lga_code',
# 'lga_name', 'ward_code', 'ward_name', 'confidence_level',
# 'coverage_percentage', 'area_km2']

In [ ]:
print(conn.execute("SELECT DISTINCT confidence_level FROM h3_cells WHERE state_name = 'Lagos' AND confidence_level IS NOT NULL").fetch_df())
# boundary_case, confident


# Example H3 cell IDs
h3_cells_to_plot =conn.execute("""SELECT DISTINCT h3_index FROM h3_cells 
                               WHERE state_name = 'Lagos' 
                               AND confidence_level IS NOT NULL
                               AND lga_name = 'Apapa'
                              --- AND ward_name = 'Abraham Adesanya'
                               """).fetch_df().h3_index.to_list() 

##### 1. Ploting with folium

In [ ]:
plot_folium = plot_h3_from_db(h3_cells_to_plot, H3_DUCKDB_PATH, show_markers=False, 
                popup_fields=['h3_derived_id', 'grid_position_id','state_name','lga_name', 'ward_name', 'confidence_level', 'coverage_percentage'
                              #,'centroid_lat', 'centroid_lng'
                              ],
                colors =  ['#3388ff']*len(h3_cells_to_plot),
                # colors = None,
                polygon_weight = 1,
                polygon_opacity = 0.05
                )

plot_folium

In [ ]:
plot_folium.save(test_map_path / 'test_save_as_folium.html')


#### 4. Using hvploting

In [ ]:
%load_ext autoreload
%autoreload 2 


import duckdb 
import geopandas as gpd
import pandas as pd
from config.settings import STORAGE_CONFIG 
from src.h3_system.plot_utils_dev import plot_h3_from_db_fast

import holoviews as hv
hv.extension('bokeh')

In [ ]:
plot_hv_bokeh = plot_h3_from_db_fast(
    h3_cell_ids = h3_cells_to_plot,
    duckdb_path = H3_DUCKDB_PATH,
    popup_fields = ['h3_derived_id', 'state_name','lga_name', 'ward_name', 'confidence_level', 'coverage_percentage'],
    color_by_column=None,
    static_fill_color='#3388ff',
    # line_color="#000000",
    show_markers=False,
    show_legend=False,
    fill_opacity=0.2,   
    width=800,
    height=600
)

plot_hv_bokeh

In [ ]:

# Save the plot to an HTML file
hv.save(plot_hv_bokeh, test_map_path/'hv_plot.html', backend='bokeh') 



### 06. SP AND CUSTOMER DATA

1. Creating Coverage Area About MFCs at Resolution 8

In [ ]:
%load_ext autoreload
%autoreload 2 

import sys
from pathlib import Path 
import geopandas as gpd

from src.get_data import DataFetcher, get_processed_data, get_geojson_data
from src.data.preprocess_data import preprocess_sp_location_mapping

In [ ]:
# 1. Fetch data from db and store locally: 
# - `sp_dim.sql` -> `df_sp_dim.feather`: Fetches stock point dimension data.
# - `sp_location_map.sql` -> `df_sp_location_mapping.feather`: Fetches the mapping of stock points to locations.
# - `get_customer_dim.sql` -> `df_customer_dim.feather`: Fetches customer dimension data.
# - `sp_active_customers.sql` -> `df_sp_active_customers.feather`: Fetches data for active customers associated with stock points.

## Fetch Data from DB
fetcher = DataFetcher(logger=logger, input_dir=str(RAW_DATA_DIR), sql_dir="_sql")
results = fetcher.fetch_all()


# ETA: 5mins

In [ ]:
%load_ext autoreload
%autoreload 2 

from src.data.preprocess_data import preprocess_sp_location_mapping, prepare_sp_and_recent_activated_customers
from src.data.preprocess_data import load_and_preprocess_sp_lga_mapping_data

# 2. Preprocess Sp Location Mapping LGA
preprocess_sp_location_mapping(logger=logger)
prepare_sp_and_recent_activated_customers(logger)

### 07. SP COVERAGE AREA AND CUSTOMER ASSIGNEMENT  


Run complete pipeline  
'''
This will return a dictionary with the following keys: ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
1. territories: A dictionary of stock point territories 
        # Dict[stock_point_id, {  
            'polygon': Union[Polygon, MultiPolygon],  
            'lga_ids': List[str],  
            'is_contiguous': bool,  
            'sub_territories': List[Polygon],  
            'total_area_km2': float,  
            'territory_version': str  
        }]  
  
2. grid_results: A dictionary of grid results for each territory  
        Dict[stock_point_id, {  
                'h3_resolution': int,  
                'h3_cells': Set[str],  
                'clipped_cells': Set[str],   
                'cell_geometries': Dict[str, Polygon],  
                'territory_coverage': float  
        }]  
          
3. assignments: A dictionary of customer assignments to stock points  
        Dict[stock_point_id, assignments_gdf with columns:  
                ['customer_id',   
                'cluster_id', 
                'h3_cell_id', 
                'assignment_confidence', 
                'assignment_tier',
                'geometry']]
        
4. optimized_clusters: A dictionary of optimized clusters for each territory

5. statistics: A dictionary of statistics for each territory

6. territory_version: The version of the territory used in the clustering
'''

In [ ]:
%load_ext autoreload
%autoreload 2 

import pandas as pd
from src.H3SpatialClusterer import H3SpatialClusterer  
from src.get_data import get_processed_data #, get_geojson_data,DataFetcher, 
import pickle
from src.utils import clean_customer_gdf_coordinates
import json
from src.utils import filter_cluster_result_dict
from src.plot_utils import plot_geojson_territory_heatmap

In [ ]:
# 3. Fetch store processed data
# lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()
lgas_gdf, sp_dim_df,  stock_point_lga_map, sp_customers_gdf, recent_customers_gdf  = get_processed_data(logger)

#### Setting Up the Pilot Stock Points

In [ ]:
pilot_2_sps = [1647402,	1647372,	1647108,	1646971,	1647109,	1647033,	
               1646999,	1647391,	1647113,	1647137,	1646991,	1647420,	
               1647141,	1647050,	1647421,	1647436,	1647380             ]
pilot_stock_point_lga_map = stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)]

#### Set-Up H3SpatialClusterer

In [ ]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df[sp_dim_df['stock_point_id'].isin(pilot_2_sps)], 
    stock_point_lga_map=stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)], 
    customers_gdf=sp_customers_gdf[sp_customers_gdf['stock_point_id'].isin(pilot_2_sps)]
)

#### EDA

In [ ]:
## Add Module src/data/preprocess_data.py
## Preprocess all sp location mapping ----------------------------------------
sp_loc_path = INPUT_BASE_DATA_SOURCES['sp_location_mapping']['local_file_path']
df_sp_location_mapping = pd.read_feather(sp_loc_path)
df_sp_location_mapping.columns = df_sp_location_mapping.columns.str.lower()
df_sp_location_mapping = (df_sp_location_mapping
                            .assign(lga_name_=lambda x: x['lga_name'].str.lower())
                            .query('~lga_name_.str.contains("self|push")', engine='python')
                            .drop(columns=['lga_name_'])
                            .reset_index(drop=True)
                            )

print(df_sp_location_mapping.columns.to_list())
print(len(df_sp_location_mapping))

## Preprocess ng lcda geometric file ----------------------------------------
import geopandas as gpd
lcda_geojson_path = ADMIN_DATA_SOURCES['wards']['standardize_file_path']
lcda_gdf = gpd.read_file(lcda_geojson_path)[[  'state_name', 'state_code','lga_name', 'lga_code','ward_name', 'ward_code']] #shape # (9410, 18)
lcda_gdf.columns = [f'{col}_ng' for col in lcda_gdf.columns]
print(lcda_gdf.columns.to_list())
print(len(lcda_gdf))

# Save to disk
# sp_dim_df.to_excel('./output/base_data_export/sp_dim.xlsx', index=False) 
# df_sp_location_mapping.to_excel('./output/base_data_export/sp_location_mapping.xlsx', index=False) 
# lcda_gdf.to_excel('./output/base_data_export/ng_wards.xlsx', index=False) 

In [ ]:
# help(pd.set_option) 

In [ ]:
# Q1. Table of Stock Point and count of LGAs mapped to them
pd.set_option("display.max_row", None)
pd.set_option("display.max_columns", None)

print(stock_point_lga_map.columns.to_list())
# print("--"*100)
# print(f"Distinct Count of LGA mapped to SPs")
# df_eda_sp_lga_count = (stock_point_lga_map.groupby(['stock_point_id', 'stock_point_name'])
#                         .agg( n_map_lgas = ('lga_id','nunique') )
#                         .sort_values('n_map_lga',ascending=False)
#                         .reset_index() 
#                         )

# print(df_eda_sp_lga_count.n_map_lgas.describe())
# print("--"*100)
# print(df_eda_sp_lga_count[['stock_point_name', 'n_map_lgas']].head(3))

# # 2. How many SP are mapped to same location (lga)
# print("--"*100)
# print(f"# 2. How many SPs were mapped to same location (lga)") 
# df_eda_lga_sp_count = (stock_point_lga_map.groupby(['state_id', 'lga_id', 'state_name', 'lga_name'])
#                         .agg(n_map_sps = ('stock_point_id','nunique') )
#                         .sort_values('n_map_sps',ascending=False)
#                         .reset_index() 
#                         )

# df_eda_sp_lga_map_sp_count = (stock_point_lga_map
#                             .merge(df_eda_lga_sp_count[['state_id', 'lga_id', 'n_map_sps']], on=['state_id', 'lga_id'], how='left')
#                             .sort_values(['n_map_sps','state_id', 'lga_id'], ascending=[False, True, True]))


# print(df_eda_lga_sp_count.n_map_sps.describe())
# print("--"*100)
# print(f'Total Number of SP with same lga mapped to another SP', df_eda_sp_lga_map_sp_count.query('n_map_sps > 1').stock_point_id.nunique(), ' Out of ',df_eda_sp_lga_map_sp_count.stock_point_id.nunique() )
# print(df_eda_lga_sp_count[['state_name', 'lga_name', 'n_map_sps']].head(3))
# print(df_eda_sp_lga_map_sp_count.merge(df_eda_lga_sp_count.iloc[0:1][['state_id','lga_id']])[['stock_point_name','state_name', 'lga_name', 'n_map_sps']])


# --------------------------------------------------------------------------------------------
print("--"*100)
print(f"Evaluating Multiple LGA Mapping to Pilot SPs")
# print(pilot_stock_point_lga_map.columns.to_list())

df_eda_lga_sp_count_pilot = (pilot_stock_point_lga_map
                                    .groupby(['state_id', 'lga_id', 'state_name', 'lga_name'])
                                    .agg(n_map_sps = ('stock_point_id','nunique') )
                                    .sort_values('n_map_sps',ascending=False)
                                    .reset_index() 
                                    )
df_eda_sp_lga_map_sp_count_pilot = (pilot_stock_point_lga_map
                                    .merge(df_eda_lga_sp_count_pilot[['state_id', 'lga_id', 'n_map_sps']], on=['state_id', 'lga_id'], how='left')
                                    .sort_values(['n_map_sps','state_id', 'lga_id'], ascending=[False, True, True])) 
print(df_eda_lga_sp_count_pilot.n_map_sps.describe())
print(df_eda_lga_sp_count_pilot.value_counts('n_map_sps'))
print("--"*100)
print(f'List of SPID with same lga mapped to another SP', df_eda_sp_lga_map_sp_count_pilot.query('n_map_sps > 1').stock_point_id.unique())
print(f'Total Number of SP with same lga mapped to another SP', df_eda_sp_lga_map_sp_count_pilot.query('n_map_sps > 1').stock_point_id.nunique(), ' Out of ',df_eda_sp_lga_map_sp_count_pilot.stock_point_id.nunique() )
print(df_eda_lga_sp_count_pilot[['state_name', 'lga_name','n_map_sps']].drop_duplicates().head(3))
print(df_eda_sp_lga_map_sp_count_pilot.merge(df_eda_lga_sp_count_pilot.iloc[0:2][['state_id','lga_id']])[['stock_point_name','state_name', 'lga_name', 'n_map_sps']])



In [ ]:
df_eda_sp_lga_map_sp_count_pilot

#### Stock Point Coverage Mapping and Clustering

In [ ]:
# Pilot sp list
pilot_sps_lists = list(set(pilot_stock_point_lga_map.stock_point_id) )
stock_point_id = str(pilot_sps_lists[1])
print('Total Pilot SPs', len(pilot_sps_lists))

In [ ]:
print(sp_customers_gdf.shape[0])
print(recent_customers_gdf.shape[0])
print(pilot_stock_point_lga_map .shape[0])

In [ ]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df[sp_dim_df['stock_point_id'].isin(pilot_2_sps)], 
    stock_point_lga_map=stock_point_lga_map[stock_point_lga_map['stock_point_id'].isin(pilot_2_sps)], 
    customers_gdf=sp_customers_gdf[sp_customers_gdf['stock_point_id'].isin(pilot_2_sps)]
)

In [ ]:
PILOT_SPS_CLUSTER_R8 =  clusterer.process_all_stock_points(territory_version="v1.2")

# Processing territory for stock point 1647380...
# 📍 Non-contiguous territory detected: 2 sub-territories

In [ ]:
# Phase 1: Territory Definition
territories = clusterer.define_territories()

# # Phase 2: H3 Grid Generation
# grid_results = self.generate_h3_grids(territories)

# # Phase 3: Customer Assignment
# assignments = self.assign_customers_to_clusters(grid_results)

In [ ]:
territories.keys()
territories['1646971']

In [ ]:
PILOT_SP_CLUSTERS_R8_PATH = EXPORTS_DIR /  "PILOT_SPS_CLUSTER_R8.pickle"

# # Save Results as pickle file
# with open(PILOT_SP_CLUSTERS_R8_PATH, 'wb') as f:
#     pickle.dump(PILOT_SPS_CLUSTER_R8, f)
 
# Testing -r8
# Open Results as pickle file
if PILOT_SP_CLUSTERS_R8_PATH.exists():
    with open(PILOT_SP_CLUSTERS_R8_PATH, 'rb') as f:
        PILOT_SPS_CLUSTER_R8 = pickle.load(f)
else:
    raise FileNotFoundError(f"Pickle file not found: {PILOT_SP_CLUSTERS_R8_PATH}")


In [ ]:
PILOT_SPS_CLUSTER_R8.keys()
# ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
# ALL_CLUSTER['territories']

In [ ]:
# # PILOT_SPS_CLUSTER['optimized_clusters']['1646991'] 
# # PILOT_SPS_CLUSTER_FLAT.keys() #['clusters', 'assignments', 'territory_summary', 'territory_cells']

# print(PILOT_SPS_CLUSTER_FLAT['clusters'].stock_point_id.nunique())
# print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())

In [ ]:
from src.utils import filter_cluster_result_dict
filtered_result = filter_cluster_result_dict(PILOT_SPS_CLUSTER_R8, pilot_sps_lists) 

filtered_result.keys()

#### Prep sharable Export File

In [ ]:
PILOT_SP_CLUSTERS_R8_PATH = EXPORTS_DIR /  "PILOT_SPS_CLUSTER_R8.pickle"
if PILOT_SP_CLUSTERS_R8_PATH.exists():
    with open(PILOT_SP_CLUSTERS_R8_PATH, 'rb') as f:
        PILOT_SPS_CLUSTER_R8 = pickle.load(f)
else:
    raise FileNotFoundError(f"Pickle file not found: {PILOT_SP_CLUSTERS_R8_PATH}")

In [ ]:
# 4. Export for deployment: ETA: 1 Min
PILOT_SPS_CLUSTER_R8_FLAT = clusterer.export_results(PILOT_SPS_CLUSTER_R8, output_format="csv") 


def extract_coverage_and_assignment_results(clusterer, cluster_result_dict):
    PILOT_SPS_CLUSTER_R8_FLAT = clusterer.export_results(cluster_result_dict, output_format="csv")

    # Stock Point Assignment Summary
    df_output_sp_coverage_cluster = PILOT_SPS_CLUSTER_R8_FLAT['territory_cells']  
    df_output_sp_customer_assignment = PILOT_SPS_CLUSTER_R8_FLAT['assignments']
    df_output_ap_coverage_cluster_summary = PILOT_SPS_CLUSTER_R8_FLAT['clusters']

    df_output_sp_customer_assignment['stock_point_id'] = df_output_sp_customer_assignment['stock_point_id'].astype(int)
    df_output_sp_coverage_cluster['stock_point_id'] = df_output_sp_coverage_cluster['stock_point_id'].astype(int)

    return df_output_sp_coverage_cluster, df_output_sp_customer_assignment, df_output_ap_coverage_cluster_summary

def prepare_sp_assignment_summary(df_output_sp_customer_assignment, sp_dim_df):
    sp_assignment_summary = (df_output_sp_customer_assignment
                            .groupby(['stock_point_id','cluster_id'])['customer_id'].count()
                            .reset_index(name='n_customers')
                            .rename({'cluster_id':'h3_cell'}, axis=1) 
                        ) 
    # Stock Point Coverage - Assignment Summary
    sp_coverage_cluster_and_assignment_summary = (df_output_sp_coverage_cluster
                                                .merge(sp_dim_df[['stock_point_id', 'stock_point_name']] , on='stock_point_id', how='left')
                                                .merge(sp_assignment_summary, how='left', on=['stock_point_id','h3_cell'])
                                                .fillna({'n_customers':0})
                                                .rename({'h3_cell':'cluster_id'}, axis=1)
                                                ) 
    sp_coverage_cluster_and_assignment_summary['n_customers'] = sp_coverage_cluster_and_assignment_summary['n_customers'].astype(int) 

    return sp_assignment_summary


df_output_sp_coverage_cluster, df_output_sp_customer_assignment, df_output_ap_coverage_cluster_summary = extract_coverage_and_assignment_results(clusterer = clusterer, 
                                                                                                                                                 cluster_result_dict = PILOT_SPS_CLUSTER_R8)
sp_coverage_cluster_and_assignment_summary = prepare_sp_assignment_summary(df_output_sp_customer_assignment, sp_dim_df)

 
print(df_output_sp_coverage_cluster.stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary.stock_point_id.nunique())

In [ ]:

## Export to DB
import duckdb 
from config.settings import STORAGE_CONFIG
from src.h3_spatial_system.storage.FastH3DuckDBManager import FastH3DuckDBManager, get_db_summary
     
             
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db: 
      # db.upsert_sp_coverage_cells(df_output_sp_coverage_cluster)
      db.upsert_customer_cluster_assignment(df_output_sp_customer_assignment)

In [ ]:
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db: 
    customer_resolution_summary = db.get_change_summary(days_back=7)
    customer_movement = db.get_customer_movements(days_back=7)
    
customer_movement
customer_resolution_summary    

In [ ]:
# df_output_sp_coverage_cluster.head(2)
# print(df_output_sp_coverage_cluster.columns.to_list()) 
# # ['h3_cell', 'stock_point_id', 'h3_resolution']

# print(df_output_sp_customer_assignment.columns.to_list()) 
# print(df_output_sp_customer_assignment.assignment_tier.value_counts())
# df_output_sp_customer_assignment.head(2)
# ['customer_id', 'cluster_id', 'h3_cell_id', 'assignment_confidence', 'assignment_tier', 'stock_point_id', 'h3_resolution'

# print(df_output_ap_coverage_cluster_summary.columns.to_list())  
# df_output_ap_coverage_cluster_summary.head(2) 
# ['cluster_id', 'h3_resolution', 'h3_cells', 'customer_count', 'parent_cluster_id', 'stock_point_id'] 

In [ ]:
import duckdb
import pandas as pd


_ = get_db_summary()

In [ ]:
conn.close()

In [ ]:
from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

In [ ]:
### Adding Coverage Clustering and Customer Assignment to db

from src.h3_spatial_system.h3_system.FastH3DuckDBManager import FastH3DuckDBManager
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
# Added New addrss columns for sp_coverage_cells
with FastH3DuckDBManager(resolution=8, db_path=H3_DUCKDB_PATH) as db: 
      db.upsert_customer_cluster_assignment(df_output_sp_customer_assignment)
      db.upsert_sp_coverage_cells(df_output_sp_coverage_cluster)
#     db._add_sp_coverage_cells_columns()


#     db._add_customer_cluster_assignment_columns()
    

In [ ]:
import duckdb 
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
conn = duckdb.connect(H3_DUCKDB_PATH)

# Install and load the httpfs extension
# conn.execute("INSTALL 'httpfs';")
conn.execute("LOAD 'httpfs';")

# Load the parquet extension if needed
conn.execute("LOAD 'parquet';")


In [ ]:
## Customer Assignment Table Enhanced
df_output_sp_customer_assignment_enhanced = conn.execute("""SELECT 
                a.stock_point_id, c.stock_point_name, a.customer_id, cluster_id, h3_derived_id cluster_code, 
                ROUND(assignment_confidence * 100, 2) assignment_confidence,
                CASE WHEN assignment_tier = 'h3_inclusion' THEN 'within cluster' 
                    WHEN assignment_tier = 'manual_review' THEN 'manual review'
                ELSE assignment_tier END AS assignment_tier,
                d.business_id,  d.contact_name, d.customer_status, kyc_capture_status, agent_id, agent_name, 
                d.state_name as customer_state_name,	
                d.town_name as customer_town_name,	
                d.city_name as customer_city_name
                FROM df_output_sp_customer_assignment a
                LEFT JOIN h3_cells b ON b.h3_index = a.cluster_id
                LEFT JOIN sp_dim_df c ON c.stock_point_id = a.stock_point_id 
                LEFT JOIN read_parquet('/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/processed/df_processed_customer_dim.parquet') d ON d.customer_id = a.customer_id             
                -- LIMIT 2
                """).df()

print(PILOT_SPS_CLUSTER_FLAT['assignments'].stock_point_id.nunique())
print(df_output_sp_customer_assignment_enhanced.stock_point_id.nunique())
df_output_sp_customer_assignment_enhanced.head(2)

In [ ]:
print(len(df_output_sp_customer_assignment))
print(len(df_output_sp_customer_assignment_enhanced))

In [ ]:
# print(conn.execute('SELECT * FROM h3_cells LIMIT 1').df().columns.to_list())
# ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 'polygon_wkt', 'boundary_json', 
#  'latlng_json', 'polygon_area', 'num_vertices', 'error', 'created_at', 'h3_derived_id', 
#  'grid_position_id', 'primary_address_id', 'country_code', 'country_name', 'state_code', 
#  'state_name', 'lga_code', 'lga_name', 'ward_code', 'ward_name', 'confidence_level', 
#  'coverage_percentage', 'area_km2']
from src.utils import calculate_distance_km
sp_coverage_cluster_and_assignment_summary_enhanced = conn.execute('''SELECT 
                    a.stock_point_id, a.stock_point_name,
                    a.cluster_id, h3_derived_id as cluster_code, 
                    a.n_customers as customer_count,
                    confidence_level as cluster_address_level, 
                    state_name as cluster_state_name, lga_name as cluster_lga_name, ward_name as cluster_ward_name,
                    --- Add coord columns if needed,
                    centroid_lat as cluster_centroid_lat, 
                    centroid_lng as cluster_centroid_lng,
                    latitude as sp_lat, 
                    longitude as sp_lng
                FROM sp_coverage_cluster_and_assignment_summary a
                LEFT JOIN  sp_dim_df b ON b.stock_point_id = a.stock_point_id
                LEFT JOIN h3_cells b ON b.h3_index = a.cluster_id 
            ''').df()

sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_km'] = (sp_coverage_cluster_and_assignment_summary_enhanced
                                                                             .apply(lambda row: calculate_distance_km(row['cluster_centroid_lat'], row['cluster_centroid_lng'], 
                                                                                                                      row['sp_lat'], row['sp_lng']), axis=1)
                                                                            )
get_cluster_status = lambda dist: 'Undefined' if dist is None else 'Within 7km' if dist <= 7 else 'Above 7km'
sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_status'] = (sp_coverage_cluster_and_assignment_summary_enhanced['cluster_sp_dist_km']
                                                                             .apply(lambda x: get_cluster_status(x))
                                                                            )    
try:
    drp_cols = [ 'cluster_centroid_lat','cluster_centroid_lng', 'sp_lat', 'sp_lng']
    sp_coverage_cluster_and_assignment_summary_enhanced.drop(drp_cols, axis=1, inplace=True)
except Exception as e:
    print(e) 


print(PILOT_SPS_CLUSTER_FLAT['territory_cells'].stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary.stock_point_id.nunique())
print(sp_coverage_cluster_and_assignment_summary_enhanced.stock_point_id.nunique())

sp_coverage_cluster_and_assignment_summary_enhanced.sample(2) 

In [ ]:
# Create a Pandas Excel writer using openpyxl as the engine
with pd.ExcelWriter(OUTPUT_DIR / 'pilot 2/coverage_cluster_and_customer_assignment.xlsx', engine='openpyxl') as writer: 
    sp_coverage_cluster_and_assignment_summary_enhanced.to_excel(writer, sheet_name='coverage_cluster', index=False)
    df_output_sp_customer_assignment_enhanced.to_excel(writer, sheet_name='assignment', index=False)

#### Export Customer Assignment to DuckDB

In [ ]:
conn = duckdb.connect(H3_DUCKDB_PATH)

conn.execute("SHOW TABLES").fetchdf() 
conn.execute("DROP TABLE IF EXISTS customer_cluster_assignment").fetchdf() 
# conn.execute("DESCRIBE customer_cluster_assignment;").fetchdf() 

print(conn.execute("SELECT COUNT(*) FROM  h3_cells").fetchone()[0] )
print(conn.execute("SELECT COUNT(*) FROM  sp_coverage_cells;").fetchone()[0])
print(conn.execute("SELECT COUNT(*) FROM  customer_cluster_assignment;").fetchone()[0])


conn.close()

# Data Migration

In [ ]:
# pip install dlt[mssql] duckdb

In [ ]:
import duckdb
from config.settings import STORAGE_CONFIG         
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path'] 

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    print(conn.execute("SHOW TABLES").fetchdf() )
    print(conn.execute("DESCRIBE stockpoint_h3_coverage; ").df())
    print(conn.execute("SELECT COUNT(*) FROM stockpoint_h3_coverage; ").df())
    df_assignment_schema = conn.execute("DESCRIBE customer_stockpoint_cluster_assignment; ").df()
    print(conn.execute("SELECT COUNT(*) FROM customer_stockpoint_cluster_assignment; ").df())
    df_h3_cells_schema = conn.execute("DESCRIBE h3_cells; ").df()
    df_h3_cells = conn.execute("""SELECT 
                                h3_index as h3_cell, resolution, centroid_lat, centroid_lng, 
                                created_at, h3_derived_id, 
                                country_code, country_name, state_code, state_name, lga_code,
                                lga_name, ward_code, ward_name, confidence_level,
                                coverage_percentage, area_km2 
                                FROM h3_cells
                                WHERE (confidence_level IS NOT NULL AND confidence_level <> 'manual_review')
                                        AND resolution=8; 
                            """).df()
    print(conn.execute("SELECT COUNT(*) FROM h3_cells; ").df())
    print(conn.execute("SELECT DISTINCT confidence_level FROM h3_cells; ").df())
    print(conn.execute("""SELECT COUNT(*) 
                       FROM h3_cells 
                       WHERE (confidence_level IS NOT NULL AND confidence_level <> 'manual_review')
                       AND resolution=8; 
                       """).df())
    

In [ ]:
# help(pd.set_option)

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
df_h3_cells.value_counts(['confidence_level'])
df_h3_cells.head(2)

In [ ]:
h3_cells_selcols = ['h3_index', 'resolution', 'centroid_lat', 'centroid_lng', 
                    'created_at', 'h3_derived_id', 
                    'country_code', 'country_name', 'state_code', 'state_name', 'lga_code',
                    'lga_name', 'ward_code', 'ward_name', 'confidence_level',
                    'coverage_percentage', 'area_km2']

In [ ]:
%load_ext autoreload
%autoreload 2 
from src.data_migration.migrate_duckdb_to_sqlsever import migrate_stockpoint_h3_coverage

In [ ]:
from src.data_migration.migrate_duckdb_to_sqlsever import migrate_stockpoint_h3_coverage
info = migrate_stockpoint_h3_coverage()

In [ ]:
%load_ext autoreload
%autoreload 2 


from src.data_migration.migrate_duckdb_to_sqlsever import (
    migrate_stockpoint_h3_coverage,
    migrate_customer_assignment, 
    migrate_h3_cells,
    migrate_all_tables,
    validate_duckdb_setup
)

# Each can be called independently
# migrate_stockpoint_h3_coverage()  # Works standalone
# migrate_customer_assignment()     # Works standalone  
# info = migrate_all_tables()             # Works standalone

In [ ]:
# Migrate h3_cells

info_h3_migration = migrate_h3_cells()

In [ ]:
info_h3_migration

# Test Data Fetch

In [1]:
%load_ext autoreload
%autoreload 2 

from config.settings import (
    ADMIN_DATA_SOURCES,
    EXPORTS_DIR,
    INPUT_BASE_DATA_SOURCES,
    OUTPUT_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    STORAGE_CONFIG,
)

from codebase.utils.utils import setup_logging
logger = setup_logging(log_dir='log-main-test-data-fetch', 
                       projname='log-main-spcr')


/home/bt/project/demand_engine/venv/lib/python3.10/site-packages/snowflake/snowpark/session.py:38: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
from src.data.get_data import DataFetcher, get_processed_data, get_geojson_data
fetcher = DataFetcher(logger=logger, input_dir=str(RAW_DATA_DIR), sql_dir="_sql")
# results = fetcher.fetch_all()
# results = fetcher.fetch_all_parallel()
# results = fetcher.fetch_all_data_via_sp()
results = fetcher.run_data_fetch()

2025-09-07 20:16:41,704 - INFO - Fetching connection string...
2025-09-07 20:16:41,705 - INFO - Connecting to database...


2025-09-07 20:16:42,537 - INFO - Executing stored procedure: usp_GetBeatAndRouteInputData
2025-09-07 20:17:19,835 - INFO - Processing result set 1: spLocationMapping
2025-09-07 20:17:20,630 - INFO - Saved 3110 rows to /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/raw/df_sp_location_mapping.parquet
2025-09-07 20:17:20,631 - INFO - Processing result set 2: spDim
2025-09-07 20:17:20,635 - INFO - Saved 78 rows to /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/raw/df_sp_dim.parquet
2025-09-07 20:17:20,636 - INFO - Processing result set 3: ActiveCustomers
2025-09-07 20:17:22,325 - INFO - Saved 31415 rows to /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/raw/df_sp_active_customers.parquet
2025-09-07 20:17:22,326 - INFO - Processing result set 4: CustomerDim
2025-09-07 20:17:56,108 - INFO - Saved 32227 rows to /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/raw/df_customer_dim.parquet
2025-09-07 20:17:5

In [9]:
import pandas as pd
path_agent_customer_mapping = RAW_DATA_DIR / 'df_agent_customer.parquet'
df_agent_customer_mapping = pd.read_parquet(path_agent_customer_mapping)

In [10]:
df_agent_customer_mapping.head(2)

,Agent_ID,Agent_Name,Role_ID,Role_Name,Customer_ID
0,5473999,Emmanuel Edum Paul,18235,Account Manager (OAM),5480472
1,5474003,Faith Michael Ekpe,18235,Account Manager (OAM),5351801


# Improve Address ID Generation

PASSED

### MAIN

In [199]:
import duckdb
from config.settings import STORAGE_CONFIG

def update_h3_location_ids(state_filter=None, h3_duckdb_path=None):
    """
    Update h3_derived_id in h3_cells table with human-readable location identifiers.
    
    Args:
        state_filter (str, optional): Filter by state name. Defaults to None (all states).
        h3_duckdb_path (str, optional): Path to H3 DuckDB file. Uses config if None.
    
    Returns:
        dict: Update statistics
    """
    if h3_duckdb_path is None:
        h3_duckdb_path = STORAGE_CONFIG['h3_duckdb_path']
    
    where_clause = f"WHERE state_name = '{state_filter}'" if state_filter else ""
    
    with duckdb.connect(h3_duckdb_path) as conn:
        # Get count before update
        before_count = conn.execute(f"SELECT COUNT(*) FROM h3_cells {where_clause}").fetchone()[0]
        
        # Create CTE with row numbers and update using it
        conn.execute(f'''
            WITH ranked_cells AS (
                SELECT 
                    h3_index,
                    CASE 
                        WHEN ward_name IS NULL OR lga_name IS NULL OR state_name IS NULL THEN NULL
                        ELSE country_code || ' | ' || 
                            TRIM(UPPER(state_name)) || ' | ' || 
                            --TRIM(UPPER(state_code)) || ' | ' || 
                            COALESCE(NULLIF(TRIM(UPPER(REPLACE(REPLACE(REPLACE(TRIM(lga_name), ' / ', '/'), 'Unknown', ''), '- ', '-'))), ''), '-') || ' | ' ||
                            COALESCE(NULLIF(TRIM(UPPER(REPLACE(REPLACE(REPLACE(TRIM(ward_name), ' / ', '/'), 'Unknown', ''), '- ', '-'))), ''), '-') || '-' ||
                            CAST(ROW_NUMBER() OVER (
                                PARTITION BY 
                                    state_name, 
                                    CASE WHEN lga_name IS NULL THEN NULL ELSE REPLACE(REPLACE(REPLACE(TRIM(lga_name), ' / ', '/'), 'Unknown', ''), '- ', '-') END,
                                    CASE WHEN ward_name IS NULL THEN NULL ELSE REPLACE(REPLACE(REPLACE(TRIM(ward_name), ' / ', '/'), 'Unknown', ''), '- ', '-') END
                                ORDER BY h3_index ASC
                            ) AS TEXT)
                    END AS new_h3_derived_id
                FROM h3_cells 
                {where_clause}
            )
            UPDATE h3_cells 
            SET h3_derived_id = ranked_cells.new_h3_derived_id
            FROM ranked_cells
            WHERE h3_cells.h3_index = ranked_cells.h3_index
        ''')
        
        # Get sample results
        sample_df = conn.execute(f'''
            SELECT h3_index, state_name, lga_name, ward_name, h3_derived_id
            FROM h3_cells 
            {where_clause}
            ORDER BY h3_index
            LIMIT 5
        ''').df()
        
        return {
            'records_updated': before_count,
            'state_filter': state_filter,
            'sample_results': sample_df
        }

# Usage examples:
if __name__ == "__main__":
    # Update Lagos only
    result = update_h3_location_ids()
    print(f"Updated {result['records_updated']} records for {result['state_filter']}")
    print("Sample results:")
    # print(result['sample_results'])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Updated 2467165 records for None
Sample results:


In [106]:
with duckdb.connect(H3_DUCKDB_PATH) as conn:
     df = conn.execute('''
        UPDATE h3_cells
        SET primary_address_id = h3_derived_id;
        ''').df()

df

,Count
0,2467165


### DEVS

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None) 

In [105]:
import duckdb
from config.settings import EXPORTS_DIR, INPUT_BASE_DATA_SOURCES, STORAGE_CONFIG
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    # RES = conn.execute('SHOW TABLES;').df()
    df = conn.execute('''
                       SELECT  
                       h3_index, state_name, lga_name, ward_name, h3_derived_id, primary_address_id                 
                       FROM h3_cells 
                       WHERE state_name = 'Ondo' 
                       ;
                       ''').df()
      
df.sample(1)

,h3_index,state_name,lga_name,ward_name,h3_derived_id,primary_address_id
15047,885888896bfffff,Ondo,Ose,Idoani 1,NG | ONDO | OSE | IDOANI 1-152,NG-ON-29017-ODSOSE02-DX9YYGV


In [ ]:
import duckdb
from config.settings import EXPORTS_DIR, INPUT_BASE_DATA_SOURCES, STORAGE_CONFIG
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    # RES = conn.execute('SHOW TABLES;').df()
    df = conn.execute('''   
                       SELECT  
                       h3_index, state_name, lga_name, ward_name, h3_derived_id                 
                       FROM h3_cells 
                       WHERE state_name = 'Lagos' 
                       ;
                       ''').df()
      
df.sample(1)

def clean_name(ward_name):
    if pd.isna(ward_name):
        return ward_name
    ward_name = ward_name.strip().title().replace(' / ', '/').replace('Unknown', '').replace('- ', '-')  
    # Add more cleaning rules as needed
    return ward_name

def make_new_id(row):
    if pd.isna(row['ward_name_cleaned']) or pd.isna(row['lga_name']) or pd.isna(row['state_name']):
        return None
    
    statename_ = row['state_name'].strip().upper()
    lganame_ = row['lga_name_cleaned'].strip().upper()
    wardname_ = row['ward_name_cleaned'].strip().upper()
    
    return f"{statename_} | {lganame_ if lganame_ != '' else '-'} | {wardname_ if wardname_ != '' else '-'}"

df['ward_name_cleaned'] = df['ward_name'].apply(clean_name)
df['lga_name_cleaned'] = df['lga_name'].apply(clean_name)
df['h3_derived_id_new'] = df.apply(make_new_id, axis=1)


df['row_number'] = df.sort_values('h3_index', ascending=True ).groupby(['state_name', 'lga_name_cleaned', 'ward_name_cleaned']).cumcount() + 1

print(df.row_number.max())
df.sample(3)

1228


,h3_index,state_name,lga_name,ward_name,h3_derived_id,ward_name_cleaned,lga_name_cleaned,h3_derived_id_new,row_number
4841,88589c92a5fffff,Lagos,Ikorodu,Owutu,NG-LA-25012-LASIKU28-YG4LKOV,Owutu,Ikorodu,LAGOS | IKORODU | OWUTU,16
6845,8858826933fffff,Lagos,Ojo,Etegbin,NG-LA-25015-LASOJO03-0J9Z01R,Etegbin,Ojo,LAGOS | OJO | ETEGBIN,6
7017,88589c9295fffff,Lagos,Ikorodu,Ajaguro,NG-LA-25012-LASIKU06-YFULZB3,Ajaguro,Ikorodu,LAGOS | IKORODU | AJAGURO,15


In [188]:
with duckdb.connect(H3_DUCKDB_PATH) as conn:
     df = conn.execute('''
        SELECT  distinct country_code,	country_name,	state_code,	state_name FROM h3_cells LIMIT 10000;
        ''').df()

df.sample(2)     

,country_code,country_name,state_code,state_name
38,NG,Nigeria,GO,Gombe
30,NG,Nigeria,EB,Ebonyi


In [ ]:
import duckdb
from config.settings import EXPORTS_DIR, INPUT_BASE_DATA_SOURCES, STORAGE_CONFIG

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    df = conn.execute('''
        SELECT  
            h3_index,
            state_name,
            lga_name,
            ward_name,
            -- h3_derived_id,            
            -- Generate new ID with row number
            CASE 
                WHEN ward_name IS NULL OR lga_name IS NULL OR state_name IS NULL THEN NULL
                ELSE country_code || ' | ' || 
                     TRIM(UPPER(state_name)) || ' | ' || 
                     --TRIM(UPPER(state_code)) || ' | ' || 
                     COALESCE(NULLIF(TRIM(UPPER(REPLACE(REPLACE(REPLACE(TRIM(lga_name), ' / ', '/'), 'Unknown', ''), '- ', '-'))), ''), '-') || ' | ' ||
                     COALESCE(NULLIF(TRIM(UPPER(REPLACE(REPLACE(REPLACE(TRIM(ward_name), ' / ', '/'), 'Unknown', ''), '- ', '-'))), ''), '-') || '-' ||
                     CAST(ROW_NUMBER() OVER (
                         PARTITION BY 
                             state_name, 
                             CASE WHEN lga_name IS NULL THEN NULL ELSE REPLACE(REPLACE(REPLACE(TRIM(lga_name), ' / ', '/'), 'Unknown', ''), '- ', '-') END,
                             CASE WHEN ward_name IS NULL THEN NULL ELSE REPLACE(REPLACE(REPLACE(TRIM(ward_name), ' / ', '/'), 'Unknown', ''), '- ', '-') END
                         ORDER BY h3_index ASC
                     ) AS TEXT)
            END AS h3_derived_id_new
            
        FROM h3_cells 
        -- WHERE state_name = 'Lagos' 
        
    ''').df()

# Sample results
df.sample(3)

,h3_index,state_name,lga_name,ward_name,h3_derived_id_new
7319,88589c8e2bfffff,Lagos,Eti Osa,Okun Ajah / Okunmopo,NG | LAGOS | ETI OSA | OKUN AJAH/OKUNMOPO-30
4973,88589cb401fffff,Lagos,Epe,Olugbokere / Abomiti,NG | LAGOS | EPE | OLUGBOKERE/ABOMITI-747
1965,885891ba85fffff,Lagos,Badagry,Apa,NG | LAGOS | BADAGRY | APA-48


In [186]:
df.sample(3)

,h3_index,state_name,lga_name,ward_name,h3_derived_id_new
8014,88589cb055fffff,Lagos,Epe,Mayunre-Oriba / Orepete-Ito Omu,NG | LA | EPE | MAYUNRE-ORIBA/OREPETE-ITO OMU-406
8178,8858836d87fffff,Lagos,Epe,Olugbokere / Abomiti,NG | LA | EPE | OLUGBOKERE/ABOMITI-587
7277,875883682ffffff,Lagos,Epe,Olugbokere / Abomiti,NG | LA | EPE | OLUGBOKERE/ABOMITI-19


-------------

# POSTPROCCESSING MAP DATA

In [97]:
import duckdb
from config.settings import EXPORTS_DIR, INPUT_BASE_DATA_SOURCES, STORAGE_CONFIG

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    # df = conn.execute(''' SHOW TABLES ''').df()
    df = conn.execute(''' SELECT * FROM stockpoint_h3_coverage WHERE cluster_sp_direction NOT NULL LIMIT 10 ''').df()
    # df = conn.execute(''' SELECT stock_point_id, customer_id , h3_resolution, COUNT(*) 
    #                   FROM customer_stockpoint_cluster_assignment 
    #                   GROUP BY stock_point_id, customer_id, h3_resolution
    #                   HAVING COUNT(*) > 1
    #                   LIMIT 10 
    #                   ''').df()
    
    
df

,id,stock_point_id,h3_cell,h3_resolution,cluster_centroid_lat,cluster_centroid_lng,cluster_sp_dist_km,cluster_sp_direction
0,5608642,1647341,8858f088abfffff,8,8.951283,7.852637,33.694397,W
1,5608643,1647442,8858d08dedfffff,8,6.325660,8.699122,38.775197,N
2,5608644,1647033,8858aa019bfffff,8,8.656669,4.889778,46.502817,SW
3,5608645,1647425,8858f28ab5fffff,8,8.116337,7.404599,96.484092,NE
4,5608646,1647421,88588a06ddfffff,8,6.201448,5.839209,29.403856,NW
5,5608647,1647341,8858f54b69fffff,8,9.261475,8.000167,61.314113,SW
6,5608648,1647421,88588a2227fffff,8,6.053879,6.009372,54.153871,NW
7,5608649,1647434,8858f68ec9fffff,8,9.036953,6.920503,20.928554,SE
8,5608650,1647033,88588525a3fffff,8,8.501406,4.366642,15.684615,E
9,5608651,1647126,8858822e89fffff,8,6.786211,3.539995,11.353293,NE


In [12]:
import duckdb
# Load customer assignments
H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']
with duckdb.connect(H3_DUCKDB_PATH) as conn: 
    customer_stockpoint_cluster_assignment_df = conn.execute('''
        SELECT 
            stock_point_id, a.customer_id, h3_cell_id, customer_type, previous_cluster_id,
            CASE WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'buying customers' THEN 1
                WHEN assignment_tier = 'manual_review' AND customer_type = 'buying customers' THEN 2
                WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'recently activated' THEN 3
            ELSE 99 END AS assignment_type_id,
            CASE WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'buying customers' THEN 'Assigned Active/Buying'
                WHEN assignment_tier = 'manual_review' AND customer_type = 'buying customers' THEN 'Unassigned Active/Buying'
                WHEN assignment_tier = 'h3_inclusion' AND customer_type = 'recently activated' THEN 'Assigned Recently Activated'
            ELSE 'Others' END AS assignment_type, 	
            contact_name, state_name, town_name, city_name, latitude, longitude, kyc_capture_status, customer_status,
            ac.agent_id, ac.agent_name, ac.role_name
        FROM customer_stockpoint_cluster_assignment a
        LEFT JOIN read_parquet('/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/processed/df_processed_customer_dim.parquet') d 
            ON d.customer_id = a.customer_id     
        LEFT JOIN df_agent_customer_mapping ac ON a.customer_id = ac.customer_id
    ''').df()

In [ ]:
# customer_stockpoint_cluster_assignment_df.head(2)

# with duckdb.connect(H3_DUCKDB_PATH) as conn: 
#     agent_customer_mfc = conn.execute('''
#                                       SELECT
#                                         stock_point_id, agent_id, agent_name, role_name, 
#                                         COUNT(DISTINCT customer_id) AS n_customers,
#                                         COUNT(DISTINCT h3_cell_id) AS n_beats,
#                                         (SELECT COUNT (DISTINCT c.customer_id) FROM customer_stockpoint_cluster_assignment_df c 
#                                             WHERE c.agent_id = a.agent_id AND c.stock_point_id = a.stock_point_id 
#                                                   AND c.assignment_type_id IN (1,2)) AS n_active_customers
#                                       FROM customer_stockpoint_cluster_assignment_df a
#                                       GROUP BY stock_point_id, agent_id, agent_name, role_name
#                                       ''').df()
     
     
     
# agent_customer_mfc.head(1)                                 

,stock_point_id,Agent_ID,Agent_Name,Role_Name,n_customers,n_beats,n_active_customers
0,1646991,5265922,Kareem Balikis Omowumi,Account Manager (OAM),30,12,18


In [1]:
from src.data.load_postprocessed_data import postprocess_map_data
postprocess_map_data(return_data = False, from_local=False)

Preparing data from scratch...
Using clustering results file: /home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/exports/clustering/ALL_SPS_CLUSTER_R8_2025-09-08.pickle


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saving processed data to local directory...
Saving processed data to cicd directory...


In [2]:
def load_from_local():
    import os 
    import gzip
    import pickle 
    from pathlib import Path
    
    map_input_dir = ci_cd_input_dir = Path('/home/bt/project/Insight_and_Discovery/Clustering_And_Routes/app/data')
    files = {
        'processed_sp_dim_df': 'processed_sp_dim_df.pkl.gz',
        'stockpoint_h3_coverage_with_metadata': 'stockpoint_h3_coverage_with_metadata.pkl.gz',
        'customer_stockpoint_cluster_assignment_df': 'customer_stockpoint_cluster_assignment_df.pkl.gz',
        'agent_customer_mfc_df': 'agent_customer_mfc.pkl.gz', 
        'sp_territories_dict': 'sp_territories_dict.pkl.gz'
    }
    
    loaded_data = {}
    for key, filename in files.items():
        filepath = map_input_dir / filename
        if not filepath.exists():
            raise FileNotFoundError(f"Local data file not found: {filepath}")
        with gzip.open(filepath, 'rb') as f:
            loaded_data[key] = pickle.load(f)
    
    return (
        loaded_data['processed_sp_dim_df'],
        loaded_data['stockpoint_h3_coverage_with_metadata'],
        loaded_data['customer_stockpoint_cluster_assignment_df'],
        loaded_data['agent_customer_mfc_df'],
        loaded_data['sp_territories_dict']
    )

In [3]:
processed_sp_dim_df, stockpoint_h3_coverage_with_metadata, \
        customer_stockpoint_cluster_assignment_df, agent_customer_mfc_df, sp_territories_dict = load_from_local()

In [26]:
# agent_customer_mfc_df.columns
stockpoint_h3_coverage_with_metadata.columns
a = stockpoint_h3_coverage_with_metadata[['latlng_coords']].iloc[0].values

In [28]:
print(a[0])

[[9.060356046720841, 7.493358135115994], [9.056063172319805, 7.492233627325159], [9.052660219594495, 7.495648289631974], [9.053549857543997, 7.500187617722963], [9.05784271008693, 7.501312491486788], [9.061245946557111, 7.497897671196086]]


In [11]:
selected_spids = [160013]
n_agents = (
        agent_customer_mfc_df[
        agent_customer_mfc_df['stock_point_id'].isin(selected_spids) #& agent_customer_mfc_df['Agent_ID'].notna()
        ].agent_id.nunique()
)

n_agents

0

In [17]:
# customer_stockpoint_cluster_assignment_df.head(2)
# customer_stockpoint_cluster_assignment_df.columns
processed_sp_dim_df[['stock_point_id', 'stock_point_name']].columns

customer_stockpoint_cluster_assignment_df_merged = (customer_stockpoint_cluster_assignment_df
    .merge(processed_sp_dim_df[['stock_point_id', 'stock_point_name']], on='stock_point_id', how='left')
    .merge(stockpoint_h3_coverage_with_metadata[['stock_point_id', 'beat','beat_id']].rename({'beat':'h3_cell_id'},axis=1), on=['stock_point_id','h3_cell_id'], how='left')
)
customer_stockpoint_cluster_assignment_df_merged.head(2)

,stock_point_id,customer_id,h3_cell_id,customer_type,previous_cluster_id,assignment_type_id,assignment_type,contact_name,state_name,town_name,city_name,latitude,longitude,kyc_capture_status,customer_status,Agent_ID,Agent_Name,Role_Name,stock_point_name,beat_id
0,1646941,5371680,8858838357fffff,buying customers,None,1,Assigned Active/Buying,Akin Akinwale,Oyo,None,Ibadan South West,7.376368,3.887569,No,Pending,5377454,W2 Olamide Abimbola Nike,Account Manager (OAM),OmniHub Ido Oyo - CARESGATE AFRICA LTD,NG | OYO | IBADAN SOUTH WEST | FOKO/OLOGEDE-1
1,1646941,5359036,885883989bfffff,recently activated,None,3,Assigned Recently Activated,Blessed Paul,Oyo,Oluyole Estreet,Ibadan South West,7.337806,3.864122,No,Pending,5295867,OLABODE TEMIDAYO W2,Account Manager (OAM),OmniHub Ido Oyo - CARESGATE AFRICA LTD,NG | OYO | OLUYOLE | ONIPE-204


In [101]:
stockpoint_h3_coverage_with_metadata.columns

Index(['stock_point_id', 'beat', 'beat_id', 'state_name', 'lga_name',
       'ward_name', 'area_km2', 'confidence_level', 'latlng_coords',
       'cluster_sp_dist_km', 'cluster_sp_direction',
       'n_total_assigned_customers', 'n_assigned_active_customers',
       'n_assigned_recent_activated_customers', 'geometry'],
      dtype='object')

In [74]:
# --- Part 1: avg customers per beat (from coverage df)
beat_stats = (
    stockpoint_h3_coverage_with_metadata
    .groupby(['stock_point_id','beat'])
    .agg(
        total_customers=('n_total_assigned_customers','sum'),
        total_active_customers=('n_assigned_active_customers','sum')
    )
    .groupby('stock_point_id')
    .mean()
    .round(0)
    .astype(int)
    .rename(columns={
        'total_customers': 'avg_customers_per_beat',
        'total_active_customers': 'avg_active_customers_per_beat'
    })
    .reset_index()
)

# --- Part 2: OAM metrics (from assignment df)

# Number of OAMs per stock_point
n_oams = (
    customer_stockpoint_cluster_assignment_df
    .groupby('stock_point_id')['Agent_ID']
    .nunique()
    .reset_index(name='n_oams')
)

# Average beats per OAM per stock_point
beats_per_oam = (
    customer_stockpoint_cluster_assignment_df
    .groupby(['stock_point_id','Agent_ID'])['h3_cell_id']
    .nunique()
    .groupby('stock_point_id')
    .mean()
    .round(0)
    .astype(int)
    .rename('beats_per_oam')
    .reset_index()
)

# Average active customers per OAM per stock_point
active_customers_per_oam = (
    customer_stockpoint_cluster_assignment_df
    .query('assignment_type_id in (1,2)')  # adjust if "active" is defined differently
    .groupby(['stock_point_id','Agent_ID'])['customer_id']
    .nunique()
    .groupby('stock_point_id')
    .mean()
    .round(0)
    .astype(int)
    .rename('active_customers_per_oam')
    .reset_index()
)

# Combine OAM stats
oam_stats = (
    n_oams
    .merge(beats_per_oam, on='stock_point_id', how='left')
    .merge(active_customers_per_oam, on='stock_point_id', how='left')
)

# --- Part 3: Merge with beat_stats
final_stats = beat_stats.merge(oam_stats, on='stock_point_id', how='left')


In [ ]:
def compute_sps_summaries(stockpoint_h3_coverage_with_metadata, customer_stockpoint_cluster_assignment_df):
    # Precompute groupings
    beat_stats = (stockpoint_h3_coverage_with_metadata
        .groupby(['stock_point_id','beat']).agg(total_customers=('n_total_assigned_customers','sum'),
            total_active_customers=('n_assigned_active_customers','sum'))
        .groupby('stock_point_id').mean().round(0).astype(int)
        .rename(columns={'total_customers': 'avg_customers_per_beat', 'total_active_customers': 'avg_active_customers_per_beat'})
        .reset_index())

    # OAM metrics
    n_oams = (customer_stockpoint_cluster_assignment_df
        .groupby('stock_point_id')['Agent_ID'].nunique().reset_index(name='n_oams'))

    beats_per_oam = (customer_stockpoint_cluster_assignment_df
        .groupby(['stock_point_id','Agent_ID'])['h3_cell_id'].nunique()
        .groupby('stock_point_id').mean().round(0).astype(int).rename('beats_per_oam').reset_index())

    active_customers_per_oam = (customer_stockpoint_cluster_assignment_df
        .query('assignment_type_id in (1,2)')
        .groupby(['stock_point_id','Agent_ID'])['customer_id'].nunique()
        .groupby('stock_point_id').mean().round(0).astype(int).rename('active_customers_per_oam').reset_index())

    oam_stats = n_oams.merge(beats_per_oam, on='stock_point_id').merge(active_customers_per_oam, on='stock_point_id')

    # Stock_point-level totals
    totals = (customer_stockpoint_cluster_assignment_df
        .groupby('stock_point_id').agg(
            total_customers=('customer_id','nunique'),
            active_customers=('customer_id', lambda x: x[customer_stockpoint_cluster_assignment_df.loc[x.index,'customer_status'] == 'Active'].nunique()),
            recently_activated=('customer_id', lambda x: x[customer_stockpoint_cluster_assignment_df.loc[x.index,'customer_type'] == 'recently activated'].nunique()),
            ).reset_index())

    total_area_and_beat = (stockpoint_h3_coverage_with_metadata
        .groupby('stock_point_id')
        .agg(
            total_area=('area_km2', 'sum'),
            total_beats=('beat', 'nunique')
        )
        .round({'total_area': 2})
        .reset_index()
    )

    # Merge everything
    final_stats = (beat_stats
        .merge(oam_stats, on='stock_point_id')
        .merge(totals, on='stock_point_id')
        .merge(total_area_and_beat, on='stock_point_id')).fillna(0)
    
    return final_stats




In [95]:
final_stats.query(' stock_point_id == 1647113')

,stock_point_id,avg_customers_per_beat,avg_active_customers_per_beat,n_oams,beats_per_oam,active_customers_per_oam
15,1647113,12,10,111,4.0,20.0


In [96]:
sps_summaries = compute_sps_summaries(stockpoint_h3_coverage_with_metadata, customer_stockpoint_cluster_assignment_df)
sps_summaries.query(' stock_point_id == 1647113')

,stock_point_id,avg_customers_per_beat,avg_active_customers_per_beat,n_oams,beats_per_oam,active_customers_per_oam,total_customers,active_customers,recently_activated,total_area,total_beats
14,1647113,12,10,111,4,20,2185,616,234,75.84,126


In [115]:
# agent_customer_mfc_df.head(2)
# agent_customer_mfc_df.sort_values('n_customers')
agent_customer_mfc_df.query('~Agent_ID.isnull()').head(2)

,stock_point_id,Agent_ID,Agent_Name,Role_Name,n_customers,n_beats,n_active_customers
0,1646941,5276192,Ogunniyi Ayodeji Emmanuel W2,Account Manager (OAM),83,26,18
1,1647402,5250989,uzor maureen chijioke,Account Manager (OAM),73,22,15


In [116]:
selected_spids = [1646941, 1647402]
agent_mfc_map = agent_customer_mfc_df[
        agent_customer_mfc_df['stock_point_id'].isin(selected_spids) & 
        agent_customer_mfc_df['Agent_ID'].notna()
    ]

In [ ]:
# agent_mfc_map.Agent_ID.nunique()

188

In [122]:
avg_customers_per_beat = sps_summaries[sps_summaries['stock_point_id'].isin(selected_spids)]['avg_customers_per_beat'].mean() if sps_summaries is not None else 0
int(avg_customers_per_beat)

1

In [123]:
avg_active_customers_per_beat = sps_summaries[sps_summaries['stock_point_id'].isin(selected_spids)]['avg_active_customers_per_beat'].mean() if sps_summaries is not None else 0
        
int(avg_active_customers_per_beat)        

0

In [124]:
avg_beats_per_oam = sps_summaries[sps_summaries['stock_point_id'].isin(selected_spids)]['beats_per_oam'].mean() if sps_summaries is not None else 0
int(avg_beats_per_oam)        

8

In [ ]:
agent_customer_mfc_df.columns
# ['stock_point_id', 'Agent_ID', 'Agent_Name', 'Role_Name', 'n_customers',
# 'n_beats', 'n_active_customers']

Index(['stock_point_id', 'Agent_ID', 'Agent_Name', 'Role_Name', 'n_customers',
       'n_beats', 'n_active_customers'],
      dtype='object')

In [128]:
selected_spids_ = [1647136]

agent_mfc_map = agent_customer_mfc_df[
    agent_customer_mfc_df['stock_point_id'].isin(selected_spids_) & 
    agent_customer_mfc_df['Agent_ID'].notna()
]

agent_mfc_map

,stock_point_id,Agent_ID,Agent_Name,Role_Name,n_customers,n_beats,n_active_customers


# Direction

In [205]:
import numpy as np
import pandas as pd

def get_simplified_direction_vectorized(lat1, lon1, lat2, lon2, detailed=False):
    """
    Vectorized calculation of simplified 8-point or detailed 16-point compass direction.
    
    Args:
        lat1 (np.array): Starting latitudes in degrees.
        lon1 (np.array): Starting longitudes in degrees.
        lat2 (np.array): Destination latitudes in degrees.
        lon2 (np.array): Destination longitudes in degrees.
        detailed (bool, optional): If True, returns 16-point directions.
                                  Defaults to False, for 8-point directions.

    Returns:
        np.array: An array of simplified or detailed direction strings.
    """
    # Convert all coordinates to radians for the bearing calculation
    lat1, lon1, lat2, lon2 = np.radians(lat1), np.radians(lon1), np.radians(lat2), np.radians(lon2)

    # Vectorized bearing formula
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.arctan2(y, x)

    # Convert bearing to degrees and normalize to a 0-360 range
    bearing_degrees = (np.degrees(bearing) + 360) % 360

    if detailed:
        # Define the 16 directions and calculate the index
        directions = np.array([
            'N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
            'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW'
        ])
        # The // operator performs vectorized floor division
        idx = (bearing_degrees + 11.25) // 22.5
        # Use the calculated indices to get the direction string
        return directions[idx.astype(int) % 16]
    else:
        # Define the 8 directions and calculate the index
        directions = np.array(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
        # The // operator performs vectorized floor division
        idx = (bearing_degrees + 22.5) // 45
        # Use the calculated indices to get the direction string
        return directions[idx.astype(int) % 8]


# --- Example Usage with a DataFrame ---

# Create a sample DataFrame
data = {
    'start_lat': [40.7128, 34.0522, 51.5074, 38.8951],
    'start_lon': [-74.0060, -118.2437, -0.1278, -77.0364],
    'end_lat': [34.0522, 40.7128, 38.8951, 51.5074],
    'end_lon': [-118.2437, -74.0060, -77.0364, -0.1278],
}
df = pd.DataFrame(data)

# Call the vectorized function for 8-point direction
df['direction_8pt'] = get_simplified_direction_vectorized(
    df['start_lat'],
    df['start_lon'],
    df['end_lat'],
    df['end_lon'],
    detailed=False
)

# Call the vectorized function for 16-point direction
df['direction_16pt'] = get_simplified_direction_vectorized(
    df['start_lat'],
    df['start_lon'],
    df['end_lat'],
    df['end_lon'],
    detailed=True
)

# Print the result
print(df)


   start_lat  start_lon  end_lat   end_lon direction_8pt direction_16pt
0    40.7128   -74.0060  34.0522 -118.2437             W              W
1    34.0522  -118.2437  40.7128  -74.0060            NE            ENE
2    51.5074    -0.1278  38.8951  -77.0364             W            WNW
3    38.8951   -77.0364  51.5074   -0.1278            NE             NE


# CUSTOMER ASSIGNMENT TRUNCATE AND INSERT TABLE 

In [ ]:
def truncate_insert_stockpoint_h3_coverage(self, df: pd.DataFrame, batch_size: int = 10000, log_changes: bool = True):
        """Truncate and insert enhanced stockpoint_h3_coverage with transaction rollback."""
        if df.empty:
            print("⚠️ No data provided")
            return
        
        print(f"📦 Truncating and inserting {len(df):,} enhanced stockpoint_h3_coverage rows...")
        self._add_stockpoint_h3_coverage_columns()
        
        required_cols = ['stock_point_id', 'h3_cell', 'h3_resolution', 'cluster_centroid_lat', 
                        'cluster_centroid_lng', 'cluster_sp_dist_km','cluster_sp_direction']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        
        try:
            self.conn.execute("BEGIN;")
            
            if log_changes:
                self.log_coverage_changes(df)
            
            self.conn.execute("TRUNCATE TABLE stockpoint_h3_coverage;")
            print("✅ Table truncated")
            
            total_inserted = 0
            for i in range(0, len(df), batch_size):
                batch_df = df.iloc[i:i+batch_size]
                self.conn.register('batch_df', batch_df)
                
                self.conn.execute("""
                    INSERT INTO stockpoint_h3_coverage 
                    (stock_point_id, h3_cell, h3_resolution, cluster_centroid_lat, 
                    cluster_centroid_lng, cluster_sp_dist_km)
                    SELECT 
                        stock_point_id, h3_cell, h3_resolution, cluster_centroid_lat,
                        cluster_centroid_lng, cluster_sp_dist_km
                    FROM batch_df
                """)
                
                self.conn.unregister('batch_df')
                total_inserted += len(batch_df)
                print(f"✅ Batch {i // batch_size + 1}: {len(batch_df):,} records inserted")
            
            self.conn.execute("COMMIT;")
            print(f"🎉 Transaction committed: {total_inserted:,} records inserted successfully")
            
        except Exception as e:
            self.conn.execute("ROLLBACK;")
            print(f"❌ Error occurred, transaction rolled back: {e}")
            try:
                self.conn.unregister('batch_df')
            except:
                pass
            raise  

# DUCKDB MIGRATION TO VM

In [5]:
import duckdb
from config.settings import EXPORTS_DIR, INPUT_BASE_DATA_SOURCES, STORAGE_CONFIG

H3_DUCKDB_PATH = STORAGE_CONFIG['h3_duckdb_path']

with duckdb.connect(H3_DUCKDB_PATH) as conn:
    # COPY TABLE TO PARAQUET WITH COMPRESSION
    conn.execute('''
        COPY h3_cells TO '/home/bt/project/demand_engine/StockPoint_Clustering_and_Routing/data/migration20250911/h3_cells.parquet' 
        (FORMAT PARQUET, COMPRESSION ZSTD);
        ''')
    # df_tables = conn.execute('''
    #     SHOW TABLES
    #     ''').df()
    

# df_tables

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,name
0,customer_stockpoint_cluster_assignment
1,h3_cells
2,stockpoint_h3_coverage
3,stockpoint_h3_coverage_log


In [ ]:
# Addtional Compression 

# --